# Train Crop Disease Detection Model (Colab, Free GPU)

This notebook trains the same model architecture used in your Flask app
(`model/model_architecture.py`), on the PlantVillage dataset, and gives you
`plant_disease_model.h5` + `class_names.json` to download and drop into
your project's `model/` folder.

**Before running:** Menu -> Runtime -> Change runtime type -> GPU.

## 1. Install kaggle + download PlantVillage dataset

In [ ]:
!pip install -q kaggle

from google.colab import files
print('Upload your kaggle.json (get it from kaggle.com -> Account -> Create New API Token)')
uploaded = files.upload()  # upload kaggle.json here

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/data --unzip

## 2. Locate train/valid folders

This dataset usually unzips to a nested folder like:
`/content/data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train`

Run the cell below to auto-detect it.

In [ ]:
import glob

train_candidates = glob.glob('/content/data/**/train', recursive=True)
valid_candidates = glob.glob('/content/data/**/valid', recursive=True)

TRAIN_DIR = train_candidates[0]
VAL_DIR = valid_candidates[0]

print('TRAIN_DIR =', TRAIN_DIR)
print('VAL_DIR   =', VAL_DIR)

## 3. Build datasets

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')

class_names = train_ds.class_names
num_classes = len(class_names)
print(num_classes, 'classes found')

normalization_layer = tf.keras.layers.Rescaling(1.0/255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])
train_ds = train_ds.map(lambda x, y: (augmentation(x, training=True), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## 4. Build model (same architecture as `model/model_architecture.py`)

In [ ]:
from tensorflow.keras import layers, models

def build_model(num_classes, input_shape=(224, 224, 3), fine_tune=False):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape, include_top=False, weights='imagenet')
    base_model.trainable = fine_tune
    if fine_tune:
        for layer in base_model.layers[:-30]:
            layer.trainable = False

    inputs = tf.keras.Input(shape=input_shape)
    x = base_model(inputs, training=fine_tune)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs)

## 5. Train (Stage 1: frozen base, Stage 2: fine-tune)

In [ ]:
model = build_model(num_classes, fine_tune=False)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('/content/plant_disease_model.h5', save_best_only=True),
]

history_1 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

In [ ]:
# Fine-tune stage (optional but improves accuracy)
model = build_model(num_classes, fine_tune=True)
model.load_weights('/content/plant_disease_model.h5')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])

history_2 = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks)
model.save('/content/plant_disease_model.h5')

## 6. Save class names + download both files

In [ ]:
import json
with open('/content/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

from google.colab import files
files.download('/content/plant_disease_model.h5')
files.download('/content/class_names.json')

## 7. Put the downloaded files in your project

Copy both downloaded files into your local project's `model/` folder
(overwrite the existing `class_names.json`), then run:

```bash
python app.py
```

The "Model not ready" message should be gone.